# GPTQ/AWQ 风格 Weight-only Quantization：激活为何重要

**面试问题：group-wise 权重量化、校准激活和显著通道保护怎样影响真实输出误差？**

## 回答主线

先明确业务目标和数据合同，再给出可比较的朴素基线；随后手写核心算法，展示中间状态、最终指标和失败路径。本 Notebook 的断言只出现在最后，用于保护关键不变量；学习重点是前面的输入、过程、对照与解释。

## 真实案例

一个三分类客服路由层要从 FP32 压到 4-bit。输入六个特征分别表示退款、物流、发票、会员、风险和金额；校准流量中“金额”通道幅度远高于其余特征。案例先做逐行 min-max 量化，再用校准激活识别显著通道并保留高精度 residual，比较分类 logits、MSE、存储账本和 OOD 失败。

### 输入预览：客服路由权重与校准激活

In [1]:
import numpy as np  # 导入矩阵运算以手写低比特权重量化。

feature_names = ["退款", "物流", "发票", "会员", "风险", "金额"]  # 定义六个具有业务语义的输入特征。
class_names = ["普通客服", "财务专席", "风控复核"]  # 定义线性层输出的三个路由类别。
weights = np.array([[0.7, 0.2, 0.3, 0.1, -0.4, 0.05], [0.4, -0.2, 1.1, 0.0, 0.2, 0.18], [-0.3, 0.1, -0.1, 0.2, 1.3, 0.42]], dtype=np.float64)  # 构造三类客服路由的 FP32 权重。
calibration = np.array([[1, 0, 0, 0, 0, 120], [0, 1, 0, 0, 0, 80], [0, 0, 1, 0, 0, 60], [0, 0, 0, 1, 0, 40], [1, 0, 0, 0, 1, 200], [0, 0, 1, 0, 1, 150]], dtype=np.float64)  # 构造金额幅度显著的校准请求。
test_inputs = np.array([[1, 0, 0, 0, 0, 100], [0, 0, 1, 0, 0, 70], [0, 0, 0, 0, 1, 180]], dtype=np.float64)  # 构造退款、发票和高风险三条可读测试请求。
print("特征顺序：", feature_names)  # 展示矩阵每一列对应的真实业务含义。
print("FP32 权重矩阵：\n", weights)  # 展示量化前的权重范围和显著列。
print("校准集每列平均绝对激活：", np.round(np.mean(np.abs(calibration), axis=0), 2))  # 展示金额通道为何需要激活感知。

特征顺序： ['退款', '物流', '发票', '会员', '风险', '金额']
FP32 权重矩阵：
 [[ 0.7   0.2   0.3   0.1  -0.4   0.05]
 [ 0.4  -0.2   1.1   0.    0.2   0.18]
 [-0.3   0.1  -0.1   0.2   1.3   0.42]]
校准集每列平均绝对激活： [  0.33   0.17   0.33   0.17   0.33 108.33]


## Baseline 基线：逐输出行对称 4-bit 量化

In [2]:
def symmetric_quantize_rows(matrix, bits=4):  # 实现逐输出行的对称均匀权重量化。
    qmax = 2 ** (bits - 1) - 1  # 计算有符号四位整数的正侧最大码值七。
    scales = np.max(np.abs(matrix), axis=1, keepdims=True) / qmax  # 用每行最大绝对值确定量化 scale。
    safe_scales = np.where(scales == 0, 1.0, scales)  # 防止全零行产生除零错误。
    codes = np.clip(np.round(matrix / safe_scales), -qmax, qmax).astype(np.int8)  # 把浮点权重舍入并裁剪到 int4 范围。
    restored = codes.astype(np.float64) * safe_scales  # 反量化得到教学推理使用的浮点近似。
    return codes, safe_scales, restored  # 返回整数码、scale 与反量化权重。

baseline_codes, baseline_scales, baseline_weights = symmetric_quantize_rows(weights)  # 对完整路由层执行朴素 4-bit 量化。
fp_outputs = test_inputs @ weights.T  # 计算三条请求的 FP32 reference logits。
baseline_outputs = test_inputs @ baseline_weights.T  # 计算朴素量化后的 logits。
baseline_mse = float(np.mean((baseline_outputs - fp_outputs) ** 2))  # 计算端到端输出均方误差。
print("朴素 int4 码值：\n", baseline_codes)  # 展示每个浮点权重实际映射到的整数。
print(f"朴素量化权重 MSE={np.mean((baseline_weights - weights) ** 2):.6f}，输出 MSE={baseline_mse:.6f}")  # 区分权重重建误差和真实输出误差。

朴素 int4 码值：
 [[ 7  2  3  1 -4  1]
 [ 3 -1  7  0  1  1]
 [-2  1 -1  1  7  2]]
朴素量化权重 MSE=0.001898，输出 MSE=28.430385


### 核心实现：用校准激活保护显著输入通道

In [3]:
def activation_aware_quantize(matrix, calibration_inputs, protected_count=1):  # 实现 AWQ 风格的显著通道保护教学版本。
    activation_importance = np.mean(np.abs(calibration_inputs), axis=0) * np.mean(np.abs(matrix), axis=0)  # 用激活幅度与权重幅度共同估计输出敏感性。
    protected = np.argsort(activation_importance)[-protected_count:]  # 选择最影响输出的输入通道保留高精度 residual。
    quantizable = matrix.copy()  # 复制权重以隔离量化主体和高精度 residual。
    residual = np.zeros_like(matrix)  # 初始化只存显著通道的稀疏 residual。
    residual[:, protected] = quantizable[:, protected]  # 把显著列写入高精度 residual。
    quantizable[:, protected] = 0.0  # 从低比特主体移除已保护列避免重复计算。
    codes, scales, restored = symmetric_quantize_rows(quantizable)  # 对剩余权重执行同样的四位量化。
    return codes, scales, restored, residual, protected, activation_importance  # 返回量化主体、残差和可解释重要性。

aware_codes, aware_scales, aware_weights, residual, protected, importance = activation_aware_quantize(weights, calibration)  # 使用真实结构校准请求执行激活感知量化。
aware_outputs = test_inputs @ (aware_weights + residual).T  # 合并低比特主体和显著列 residual 计算 logits。
aware_mse = float(np.mean((aware_outputs - fp_outputs) ** 2))  # 计算激活感知方案的端到端输出误差。
print("输入通道重要性：", {name: round(score, 3) for name, score in zip(feature_names, importance)})  # 展示为何金额通道被选中。
print("受保护通道：", [feature_names[index] for index in protected])  # 输出高精度 residual 的实际业务字段。
print(f"激活感知输出 MSE={aware_mse:.6f}，相对朴素方案改善={(baseline_mse - aware_mse) / baseline_mse:.1%}")  # 量化 AWQ 风格保护带来的收益。

输入通道重要性： {'退款': 0.156, '物流': 0.028, '发票': 0.167, '会员': 0.017, '风险': 0.211, '金额': 23.472}
受保护通道： ['金额']
激活感知输出 MSE=0.002154，相对朴素方案改善=100.0%


## 结果解读：逐请求 logits 与路由是否改变

In [4]:
request_names = ["百元退款", "补开发票", "高额风险"]  # 给三条测试向量附上可读请求名称。
print("请求          FP32路由   朴素int4路由  激活感知路由  最大logit误差")  # 输出逐请求对照表表头。
for name, reference, naive, aware in zip(request_names, fp_outputs, baseline_outputs, aware_outputs):  # 对每条请求比较三种 logits 和最终类别。
    reference_class = class_names[int(np.argmax(reference))]  # 获取 FP32 reference 的路由类别。
    naive_class = class_names[int(np.argmax(naive))]  # 获取朴素 int4 的路由类别。
    aware_class = class_names[int(np.argmax(aware))]  # 获取激活感知方案的路由类别。
    max_error = float(np.max(np.abs(aware - reference)))  # 计算该请求最大的 logit 偏差。
    print(f"{name:<12} {reference_class:<10} {naive_class:<12} {aware_class:<12} {max_error:8.4f}")  # 展示误差是否真正影响业务决策。
print("解读：权重 MSE 只是代理指标；校准目标应优先关注真实激活下的 logits、任务质量和错误切片。")  # 明确 GPTQ/AWQ 工程验收不应停留在权重矩阵。

请求          FP32路由   朴素int4路由  激活感知路由  最大logit误差
百元退款         风控复核       风控复核         风控复核           0.0714
补开发票         风控复核       风控复核         风控复核           0.0857
高额风险         风控复核       风控复核         风控复核           0.0429
解读：权重 MSE 只是代理指标；校准目标应优先关注真实激活下的 logits、任务质量和错误切片。


## 失败案例：校准集没有覆盖新的高幅度风险通道

In [5]:
ood_inputs = np.array([[0, 0, 0, 0, 250, 1], [0, 0, 0, 0, 320, 0]], dtype=np.float64)  # 构造风险通道突增而金额很小的分布外流量。
ood_reference = ood_inputs @ weights.T  # 计算 OOD 请求的 FP32 reference logits。
ood_aware = ood_inputs @ (aware_weights + residual).T  # 使用只保护金额列的旧量化制品推理。
ood_error = float(np.mean((ood_aware - ood_reference) ** 2))  # 计算校准分布漂移后的输出误差。
recalibration = np.vstack([calibration, ood_inputs])  # 把新风险流量加入校准快照。
_, _, refreshed_weights, refreshed_residual, refreshed_protected, _ = activation_aware_quantize(weights, recalibration, protected_count=2)  # 重新选择两个显著通道。
refreshed_outputs = ood_inputs @ (refreshed_weights + refreshed_residual).T  # 使用刷新后的量化制品重新推理。
refreshed_error = float(np.mean((refreshed_outputs - ood_reference) ** 2))  # 计算重新校准后的 OOD 输出误差。
print(f"旧校准 OOD MSE={ood_error:.6f}，重新校准 MSE={refreshed_error:.6f}")  # 展示分布漂移会让旧保护策略失效。
print("重新校准保护通道：", [feature_names[index] for index in refreshed_protected])  # 展示风险和金额两列都应被保护。

旧校准 OOD MSE=50.479592，重新校准 MSE=0.000000
重新校准保护通道： ['金额', '风险']


### 生产边界

In [6]:
fp_bytes = weights.size * 4  # 估算 FP32 权重主体的字节数。
teaching_int4_bytes = int(np.ceil(aware_codes.size / 2)) + aware_scales.size * 4 + residual[:, protected].size * 4  # 估算打包 int4、scale 和显著 residual 的真实存储。
artifact = {"bits": 4, "axis": "output-row", "protected_features": [feature_names[index] for index in protected], "calibration_snapshot": "support-traffic-2026w30", "estimated_bytes": teaching_int4_bytes, "fp32_bytes": fp_bytes}  # 构造量化制品必须携带的元数据。
print("量化制品账本：", artifact)  # 展示不能只保存整数权重而丢失 scale、校准和 residual。
print("生产替换点：真实 GPTQ 还使用二阶误差补偿，真实 AWQ 结合通道缩放与高效 kernel；两者都需端到端质量、延迟和峰值显存验收。")  # 明确教学保护列方案与论文系统的差距。

量化制品账本： {'bits': 4, 'axis': 'output-row', 'protected_features': ['金额'], 'calibration_snapshot': 'support-traffic-2026w30', 'estimated_bytes': 33, 'fp32_bytes': 72}
生产替换点：真实 GPTQ 还使用二阶误差补偿，真实 AWQ 结合通道缩放与高效 kernel；两者都需端到端质量、延迟和峰值显存验收。


## 回归测试：只保护码值、收益和漂移探针

In [7]:
assert aware_codes.min() >= -7 and aware_codes.max() <= 7  # 验证四位对称量化码值没有越界。
assert protected.tolist() == [5]  # 验证当前校准快照正确识别金额显著通道。
assert aware_mse < baseline_mse  # 验证激活感知保护在目标流量上优于朴素基线。
assert refreshed_error < ood_error  # 验证加入风险流量后重新校准能够修正 OOD 误差。
assert teaching_int4_bytes < fp_bytes  # 验证加入 scale 和 residual 后仍获得实际存储收益。
print("回归测试通过：int4 范围、显著通道、目标收益、漂移修正和真实字节账本均成立。")  # 用少量断言保护最重要的量化合同。

回归测试通过：int4 范围、显著通道、目标收益、漂移修正和真实字节账本均成立。
